# 4. Classificação pelos eixos da BNCC
A classificação é baseada na presença de termos e expressões no texto normalizado. Um artigo pode receber mais de um eixo. Todas as evidências serão registradas para revisão.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / 'README.md').exists() and (p / 'dados').exists()), None)
if raiz is None:
    raise FileNotFoundError('Não foi possível localizar a pasta do projeto.')
sys.path.insert(0, str(raiz))

from apoio.funcoes import carregar_eixos_bncc, selecionar_descritores_relevantes, encontrar_evidencias_bncc
pasta_processados = raiz / 'dados' / '1_processados'

## 4.1 Leitura dos artigos e do vocabulário

In [ ]:
# Carregar os dados processados
df = pd.read_csv(pasta_processados / '02_artigos_pre_processados.csv', encoding='utf-8-sig')
ranking_termos = pd.read_csv(pasta_processados / '03_ranking_termos_titulos.csv', encoding='utf-8-sig')
ranking_bigramas = pd.read_csv(pasta_processados / '03_ranking_bigramas_titulos.csv', encoding='utf-8-sig')
eixos_candidatos = carregar_eixos_bncc(raiz / 'apoio' / 'termos_bncc.yml')

if 'evento' not in df.columns:
    raise ValueError("A coluna 'evento' é obrigatória para a classificação segmentada por evento.")

# Seleciona descritores relevantes para cada evento, em vez de usar um vocabulário global

def selecionar_eixos_por_evento(evento: str) -> dict[str, dict[str, object]]:
    termos_evento = set(ranking_termos.loc[ranking_termos['evento'] == evento, 'termo'])
    bigramas_evento = set(ranking_bigramas.loc[ranking_bigramas['evento'] == evento, 'bigrama'])
    return selecionar_descritores_relevantes(eixos_candidatos, termos_evento, bigramas_evento)

# Mantém o vocabulário de cada evento para a classificação
vocabularios_por_evento = {
    evento: selecionar_eixos_por_evento(evento)
    for evento in sorted(df['evento'].dropna().astype(str).str.strip().unique())
}

print(f'Artigos recebidos: {len(df)}')
print('Eventos disponíveis:', sorted(vocabularios_por_evento.keys()))

# Criar DataFrame com os resultados por evento
pd.DataFrame([
    {'evento': evento, 'codigo': codigo, 'eixo': dados['nome'], 'termos_ativos': len(dados['termos']), 'bigramas_ativos': len(dados['bigramas'])}
    for evento, eixos in vocabularios_por_evento.items()
    for codigo, dados in eixos.items()
])

## 4.2 Aplicação das regras
A busca usa palavras e expressões completas. Por exemplo, o termo `tic` não será encontrado dentro de outra palavra.

In [ ]:
# Classifica cada artigo usando o vocabulário específico do seu evento

def classificar_artigo_por_evento(linha: pd.Series) -> dict[str, dict[str, list[str]]]:
    evento = str(linha['evento']).strip()
    eixos_evento = vocabularios_por_evento.get(evento, {})
    return encontrar_evidencias_bncc(str(linha['texto_limpo']).strip(), eixos_evento)


def listar_nomes_eixos(resultado: dict[str, dict[str, list[str]]], evento: str) -> str:
    eixos_evento = vocabularios_por_evento.get(evento, {})
    return '; '.join(eixos_evento[codigo]['nome'] for codigo in resultado)


df['evidencias_bncc'] = df.apply(classificar_artigo_por_evento, axis=1)
df['eixos_bncc'] = df.apply(
    lambda linha: listar_nomes_eixos(linha['evidencias_bncc'], str(linha['evento']).strip()),
    axis=1,
)
df['quantidade_eixos'] = df['evidencias_bncc'].apply(len)

## 4.3 Tabela de classificações e evidências

In [ ]:
linhas_classificacao = []
# Gerar linhas de classificação para cada artigo e cada eixo
for _, artigo in df.iterrows():
    for codigo_eixo, evidencias in artigo['evidencias_bncc'].items():
        linhas_classificacao.append({
            'id_artigo': artigo['id_artigo'],
            'evento': artigo['evento'],
            'ano': artigo['ano'],
            'eixo_bncc': vocabularios_por_evento[str(artigo['evento']).strip()][codigo_eixo]['nome'],
            'quantidade_evidencias': len(evidencias['termos']) + len(evidencias['bigramas']),
            'termos_encontrados': ' | '.join(evidencias['termos']),
            'bigramas_encontrados': ' | '.join(evidencias['bigramas']),
        })

# Criar DataFrame com as classificações
classificacoes = pd.DataFrame(linhas_classificacao)

## 4.7 Exportação para validação

In [ ]:
artigos_classificados = df.drop(columns='evidencias_bncc')

artigos_classificados.to_csv(pasta_processados / '04_artigos_classificados_bncc.csv', index=False, encoding='utf-8-sig')
classificacoes.to_csv(pasta_processados / '04_classificacoes_bncc.csv', index=False, encoding='utf-8-sig')

print('Arquivos necessários à geração do Excel exportados.')